In [2]:
!pip install optuna

In [3]:
import optuna
import joblib
import numpy as np
import pandas as pd

from xgboost import XGBClassifier

from sklearn.metrics import *

import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
from pathlib import Path

DATA = Path("../Data")

train = pd.read_csv(DATA/"train_2000_2020.csv")
validation = pd.read_csv(DATA/"validation_2021_2022.csv")
test = pd.read_csv(DATA/"test_2023_2025.csv")

DROP = [
    "Date",
    "Time",
    "Thunderstorm_24h"
]

X_train = train.drop(columns=DROP)
y_train = train["Thunderstorm_24h"]

X_validation = validation.drop(columns=DROP)
y_validation = validation["Thunderstorm_24h"]

X_test = test.drop(columns=DROP)
y_test = test["Thunderstorm_24h"]

In [5]:
def weather_metrics(y_true, y_pred):

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    precision = tp/(tp+fp) if tp+fp else 0

    recall = tp/(tp+fn) if tp+fn else 0

    far = fp/(tp+fp) if tp+fp else 0

    csi = tp/(tp+fp+fn) if tp+fp+fn else 0

    f1 = f1_score(y_true,y_pred)

    return precision, recall, far, csi, f1

In [6]:
weights = [

1,

1.2,

1.5,

2,

2.5,

3,

4,

5

]

In [7]:
best_score = -1

best_model = None

best_params = None

best_threshold = None

best_weight = None

In [8]:
for weight in weights:

    print("="*60)

    print("Testing scale_pos_weight =",weight)

    def objective(trial):

        model = XGBClassifier(

            objective="binary:logistic",

            tree_method="hist",

            eval_metric="logloss",

            learning_rate=trial.suggest_float(
                "learning_rate",
                0.01,
                0.15,
                log=True
            ),

            max_depth=trial.suggest_int(
                "max_depth",
                4,
                10
            ),

            n_estimators=trial.suggest_int(
                "n_estimators",
                500,
                3000
            ),

            min_child_weight=trial.suggest_int(
                "min_child_weight",
                1,
                10
            ),

            gamma=trial.suggest_float(
                "gamma",
                0,
                5
            ),

            subsample=trial.suggest_float(
                "subsample",
                0.6,
                1.0
            ),

            colsample_bytree=trial.suggest_float(
                "colsample_bytree",
                0.6,
                1.0
            ),

            reg_alpha=trial.suggest_float(
                "reg_alpha",
                0,
                2
            ),

            reg_lambda=trial.suggest_float(
                "reg_lambda",
                0,
                5
            ),

            scale_pos_weight=weight,

            random_state=42
        )

        model.fit(

            X_train,

            y_train,

            eval_set=[

                (X_validation,y_validation)

            ],

            verbose=False

        )

        probability = model.predict_proba(
            X_validation
        )[:,1]

        best_local = -1

        for threshold in np.arange(0.20,0.81,0.01):

            prediction = (
                probability>=threshold
            ).astype(int)

            precision,recall,far,csi,f1 = weather_metrics(

                y_validation,

                prediction

            )

            if recall < 0.70:

                continue

            if csi > best_local:

                best_local = csi

        return best_local

Testing scale_pos_weight = 1
Testing scale_pos_weight = 1.2
Testing scale_pos_weight = 1.5
Testing scale_pos_weight = 2
Testing scale_pos_weight = 2.5
Testing scale_pos_weight = 3
Testing scale_pos_weight = 4
Testing scale_pos_weight = 5


In [19]:
study = optuna.create_study(
        direction="maximize"
    )

study.optimize(
        objective,
        n_trials=100
    )

[I 2026-07-14 22:29:42,128] A new study created in memory with name: no-name-b3d6206a-5b00-4c73-ba2a-e9dda93d036a
[I 2026-07-14 22:29:46,348] Trial 0 finished with value: 0.3743016759776536 and parameters: {'learning_rate': 0.048016759154076286, 'max_depth': 4, 'n_estimators': 700, 'min_child_weight': 4, 'gamma': 2.112521228432529, 'subsample': 0.9572512309141138, 'colsample_bytree': 0.8893575877315785, 'reg_alpha': 0.09523192729213781, 'reg_lambda': 4.509972714261553}. Best is trial 0 with value: 0.3743016759776536.
[I 2026-07-14 22:29:55,234] Trial 1 finished with value: 0.378698224852071 and parameters: {'learning_rate': 0.013351044606199874, 'max_depth': 7, 'n_estimators': 597, 'min_child_weight': 4, 'gamma': 0.7416612638638825, 'subsample': 0.9223919223526744, 'colsample_bytree': 0.90626449702423, 'reg_alpha': 1.5247013418151416, 'reg_lambda': 2.659522065396434}. Best is trial 1 with value: 0.378698224852071.
[I 2026-07-14 22:30:09,049] Trial 2 finished with value: 0.3192488262910

In [22]:
model = XGBClassifier(

        **study.best_params,

        objective="binary:logistic",

        tree_method="hist",

        eval_metric="logloss",

        scale_pos_weight=weight,

        random_state=42

    )

model.fit(

        X_train,

        y_train,

        eval_set=[

            (X_validation,y_validation)

        ],

        verbose=False
    )

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8229492038542171
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [29]:
probability = model.predict_proba(X_validation)[:, 1]

for threshold in np.arange(0.20, 0.81, 0.01):

    prediction = (probability >= threshold).astype(int)

    precision, recall, far, csi, f1 = weather_metrics(
        y_validation,
        prediction
    )

    if recall < 0.70:
        continue

    if csi > best_score:

        best_score = csi
        best_model = model
        best_threshold = threshold
        best_weight = weight
        best_params = study.best_params

In [30]:
print("="*60)

print("BEST MODEL")

print("="*60)

print(best_weight)

print(best_threshold)

print(best_score)

print(best_params)

BEST MODEL
5
0.6000000000000003
0.3950617283950617
{'learning_rate': 0.022642660336699563, 'max_depth': 10, 'n_estimators': 2748, 'min_child_weight': 2, 'gamma': 4.165110701427644, 'subsample': 0.9312685233449287, 'colsample_bytree': 0.8229492038542171, 'reg_alpha': 1.957978838122957, 'reg_lambda': 0.40529687437446316}


In [33]:
val_probability = best_model.predict_proba(X_validation)[:, 1]
val_prediction = (val_probability >= best_threshold).astype(int)

print("=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)

print(f"Accuracy  : {accuracy_score(y_validation, val_prediction):.4f}")
print(f"Precision : {precision_score(y_validation, val_prediction):.4f}")
print(f"Recall    : {recall_score(y_validation, val_prediction):.4f}")
print(f"F1 Score  : {f1_score(y_validation, val_prediction):.4f}")
print(f"ROC AUC   : {roc_auc_score(y_validation, val_probability):.4f}")

print(confusion_matrix(y_validation, val_prediction))
print(classification_report(y_validation, val_prediction))

VALIDATION RESULTS
Accuracy  : 0.8310
Precision : 0.4741
Recall    : 0.7033
F1 Score  : 0.5664
ROC AUC   : 0.8577
[[418  71]
 [ 27  64]]
              precision    recall  f1-score   support

           0       0.94      0.85      0.90       489
           1       0.47      0.70      0.57        91

    accuracy                           0.83       580
   macro avg       0.71      0.78      0.73       580
weighted avg       0.87      0.83      0.84       580



In [34]:
# ================================
# Final Testing
# ================================

probability = best_model.predict_proba(X_test)[:, 1]

prediction = (probability >= best_threshold).astype(int)

# Metrics
accuracy = accuracy_score(y_test, prediction)
precision = precision_score(y_test, prediction, zero_division=0)
recall = recall_score(y_test, prediction, zero_division=0)
f1 = f1_score(y_test, prediction, zero_division=0)
roc = roc_auc_score(y_test, probability)

tn, fp, fn, tp = confusion_matrix(y_test, prediction).ravel()

far = fp / (tp + fp) if (tp + fp) > 0 else 0
csi = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
balanced_accuracy = (recall + specificity) / 2

print("=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print(f"Accuracy           : {accuracy:.4f}")
print(f"Precision          : {precision:.4f}")
print(f"Recall (POD)       : {recall:.4f}")
print(f"Specificity        : {specificity:.4f}")
print(f"Balanced Accuracy  : {balanced_accuracy:.4f}")
print(f"F1 Score           : {f1:.4f}")
print(f"ROC AUC            : {roc:.4f}")
print(f"False Alarm Ratio  : {far:.4f}")
print(f"CSI (Threat Score) : {csi:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, prediction))

print("\nClassification Report")
print(classification_report(y_test, prediction))

FINAL TEST RESULTS
Accuracy           : 0.7823
Precision          : 0.3132
Recall (POD)       : 0.6301
Specificity        : 0.8039
Balanced Accuracy  : 0.7170
F1 Score           : 0.4184
ROC AUC            : 0.8071
False Alarm Ratio  : 0.6868
CSI (Threat Score) : 0.2646

Confusion Matrix
[[980 239]
 [ 64 109]]

Classification Report
              precision    recall  f1-score   support

           0       0.94      0.80      0.87      1219
           1       0.31      0.63      0.42       173

    accuracy                           0.78      1392
   macro avg       0.63      0.72      0.64      1392
weighted avg       0.86      0.78      0.81      1392



In [32]:
joblib.dump(

best_model,

DATA/"XGBoost_Final.pkl"

)

print("Saved Successfully")

Saved Successfully


In [1]:
print(best_thr)

NameError: name 'best_thr' is not defined